In [ ]:
#규칙기반 가중치를 이용한 방법
#사용자 상태에 따라 표의 가중치를 자동으로 선택

import copy
import json

import numpy as np
import pandas as pd


# ============================================================
# 1. 추천 설정
# ============================================================

# 최근 30일, 서로 다른 표준 상품(product_id) 수 기준
#
# 기본 사용자:
#   상세 조회 상품 >= 5개
#   AND 유효 예약·확인 구매 상품 합집합 >= 5개
#
# 관심 행동 사용자:
#   기본 사용자 조건 미충족
#   AND 찜·유효 예약 상품 합집합 >= 2개
#
# 초기 사용자:
#   나머지
#
# 관심 상품을 직접 선택한 것만으로 사용자 상태가 바뀌지는 않습니다.

RULE_VERSION = "rules_v1"

WEIGHTS = {
    "new_no_interest": {
        "interest_match": 0.00,
        "category_preference": 0.00,
        "store_preference": 0.00,
        "distance_score": 0.75,
        "recency_score": 0.25,
    },
    "new_with_interest": {
        "interest_match": 0.60,
        "category_preference": 0.00,
        "store_preference": 0.00,
        "distance_score": 0.30,
        "recency_score": 0.10,
    },
    "behavior_based": {
        "interest_match": 0.40,
        "category_preference": 0.25,
        "store_preference": 0.15,
        "distance_score": 0.15,
        "recency_score": 0.05,
    },
}

STATE_NAMES = {
    "new_no_interest": "초기 사용자: 행동 조건 미충족",
    "new_with_interest": "관심 행동 사용자: 찜·유효 예약 상품 2개 이상",
    "behavior_based": "기본 사용자: 조회·예약·구매 조건 충족",
}

ACTION_POINTS = {
    "detail_view": 1.0,
    "favorite": 3.0,
    "reserve": 5.0,
    "purchase_confirmed": 8.0,
}

CANDIDATE_COLUMNS = [
    "listing_id",
    "product_id",
    "name",
    "category",
    "store_id",
    "distance_km",
    "hours_since_created",
]


# ============================================================
# 2. 공통 입력값 처리
# ============================================================

def normalize_id(value):
    """ID를 문자열로 통일합니다."""
    if value is None or pd.isna(value):
        raise ValueError("ID가 비어 있습니다.")

    result = str(value).strip()

    if not result:
        raise ValueError("ID가 비어 있습니다.")

    return result


def parse_aware_time(value):
    """시간대가 포함된 시각을 한국 시간으로 변환합니다."""
    if value is None:
        raise ValueError("시각이 비어 있습니다.")

    timestamp = pd.Timestamp(value)

    if pd.isna(timestamp) or timestamp.tzinfo is None:
        raise ValueError(
            "시각에는 시간대가 필요합니다. "
            "예: 2026-11-03T15:00:00+09:00"
        )

    return timestamp.tz_convert("Asia/Seoul")


def resolve_now(now=None):
    if now is None:
        return pd.Timestamp.now(tz="Asia/Seoul")

    return parse_aware_time(now)


# ============================================================
# 3. 최근 30일의 유효한 행동 조회
# ============================================================

def get_recent_valid_events(events, now=None):
    """
    - 최근 30일 기록만 사용
    - 미래 기록 제외
    - 동일 event_id 중복 제거
    - 무효 예약 제외

    구매는 백엔드에서 확인한 purchase_confirmed만 전달합니다.
    """
    now = resolve_now(now)
    cutoff = now - pd.Timedelta(days=30)

    rows = []
    seen_event_ids = set()

    for raw_event in events:
        event_id = normalize_id(raw_event["event_id"])

        if event_id in seen_event_ids:
            continue

        seen_event_ids.add(event_id)

        event_type = raw_event["event_type"]

        if event_type not in ACTION_POINTS:
            raise ValueError(
                f"지원하지 않는 행동 종류: {event_type}"
            )

        occurred_at = parse_aware_time(
            raw_event["occurred_at"]
        )

        if not cutoff <= occurred_at <= now:
            continue

        if event_type == "reserve":
            is_valid = raw_event["is_valid_reservation"]

            if not isinstance(is_valid, bool):
                raise ValueError(
                    "is_valid_reservation은 "
                    "True 또는 False여야 합니다."
                )

            if not is_valid:
                continue

        rows.append({
            "event_id": event_id,
            "product_id": normalize_id(
                raw_event["product_id"]
            ),
            "store_id": normalize_id(
                raw_event["store_id"]
            ),
            "event_type": event_type,
            "occurred_at": occurred_at,
            "is_valid_reservation": event_type == "reserve",
        })

    return pd.DataFrame(
        rows,
        columns=[
            "event_id",
            "product_id",
            "store_id",
            "event_type",
            "occurred_at",
            "is_valid_reservation",
        ],
    )


# ============================================================
# 4. 사용자 상태 및 관심 상품 계산
# ============================================================

def get_user_recommendation_profile(user, now=None):
    recent = get_recent_valid_events(
        user.get("events", []),
        now=now,
    )

    def product_ids_for(event_types):
        return set(
            recent.loc[
                recent["event_type"].isin(event_types),
                "product_id",
            ]
        )

    viewed_ids = product_ids_for(["detail_view"])

    favorite_or_reserved_ids = product_ids_for([
        "favorite",
        "reserve",
    ])

    reserved_or_purchased_ids = product_ids_for([
        "reserve",
        "purchase_confirmed",
    ])

    # 우선순위: 기본 → 관심 행동 → 초기 사용자
    if (
        len(viewed_ids) >= 5
        and len(reserved_or_purchased_ids) >= 5
    ):
        state = "behavior_based"

    elif len(favorite_or_reserved_ids) >= 2:
        state = "new_with_interest"

    else:
        state = "new_no_interest"

    # 명시적 관심 상품 + 최근 행동으로 추정한 관심 상품
    # 상세 조회만 한 상품은 관심 상품 집합에 넣지 않습니다.
    explicit_interest_ids = {
        normalize_id(product_id)
        for product_id in user.get(
            "interest_product_ids", []
        )
    }

    inferred_interest_ids = product_ids_for([
        "favorite",
        "reserve",
        "purchase_confirmed",
    ])

    return {
        "state": state,
        "interest_product_ids": (
            explicit_interest_ids | inferred_interest_ids
        ),
        "viewed_product_count": len(viewed_ids),
        "favorite_or_reserved_product_count": len(
            favorite_or_reserved_ids
        ),
        "reserved_or_purchased_product_count": len(
            reserved_or_purchased_ids
        ),
    }


def get_user_state(user, now=None):
    return get_user_recommendation_profile(
        user,
        now=now,
    )["state"]


# ============================================================
# 5. 카테고리 선호도 자동 계산
# ============================================================

def calculate_category_preferences(user, now=None):
    product_categories = user["product_categories"]

    recent = get_recent_valid_events(
        user.get("events", []),
        now=now,
    )

    # 직접 선택한 관심 상품은 상품당 3점
    product_scores = {
        normalize_id(product_id): 3.0
        for product_id in user.get(
            "interest_product_ids", []
        )
    }

    # 같은 상품은 가장 높은 행동 점수 하나만 반영
    for event in recent.to_dict(orient="records"):
        product_id = event["product_id"]
        points = ACTION_POINTS[event["event_type"]]

        product_scores[product_id] = max(
            product_scores.get(product_id, 0.0),
            points,
        )

    category_scores = {}

    for product_id, points in product_scores.items():
        category = product_categories.get(product_id)

        if category is None:
            raise ValueError(
                f"상품 {product_id}의 카테고리 정보가 없습니다."
            )

        category_scores[category] = (
            category_scores.get(category, 0.0) + points
        )

    total = sum(category_scores.values())

    if total == 0:
        return {}

    # 카테고리 선호도의 합계가 1이 되도록 변환
    return {
        category: score / total
        for category, score in category_scores.items()
    }

# ============================================================
# 5-1. 점포 선호도 자동 계산
# ============================================================

def calculate_store_preferences(user, now=None):
    recent = get_recent_valid_events(
        user.get("events", []),
        now=now,
    )

    if recent.empty:
        return {}

    # 같은 점포·같은 상품에서는 가장 높은 행동 점수만 인정
    product_scores = {}

    for event in recent.to_dict(orient="records"):
        key = (
            event["store_id"],
            event["product_id"],
        )

        points = ACTION_POINTS[event["event_type"]]

        product_scores[key] = max(
            product_scores.get(key, 0.0),
            points,
        )

    # 점포별 합산
    store_scores = {}

    for (store_id, product_id), points in product_scores.items():
        store_scores[store_id] = (
            store_scores.get(store_id, 0.0) + points
        )

    total = sum(store_scores.values())

    if total == 0:
        return {}

    return {
        store_id: score / total
        for store_id, score in store_scores.items()
    }

# ============================================================
# 6. 추천 점수 계산 및 정렬
# ============================================================

def recommend(user, candidates, top_n=None, now=None):
    """
    candidates는 백엔드에서 판매 상태·시장·거리 범위를
    확인한 후보여야 합니다.

    점포 선호도는 최근 30일의 유효 행동으로 자동 계산합니다.
    """
    now = resolve_now(now)

    missing_columns = [
        column
        for column in CANDIDATE_COLUMNS
        if column not in candidates.columns
    ]

    if missing_columns:
        raise ValueError(
            f"상품 데이터에 없는 컬럼: {missing_columns}"
        )

    if top_n is not None and (
        isinstance(top_n, bool)
        or not isinstance(top_n, int)
        or top_n < 1
    ):
        raise ValueError("top_n은 1 이상의 정수여야 합니다.")

    df = candidates[CANDIDATE_COLUMNS].copy()

    # 점수 계산에 사용하는 숫자 검증
    for column in ["distance_km", "hours_since_created"]:
        df[column] = pd.to_numeric(
            df[column],
            errors="raise",
        ).astype(float)

        if (
            not np.isfinite(df[column].to_numpy()).all()
            or (df[column] < 0).any()
        ):
            raise ValueError(
                f"{column}에는 결측값 없이 "
                "0 이상의 숫자를 넣어 주세요."
            )

    profile = get_user_recommendation_profile(
        user,
        now=now,
    )

    state = profile["state"]
    weights = WEIGHTS[state].copy()

    category_preferences = calculate_category_preferences(
        user,
        now=now,
    )

    store_preferences = calculate_store_preferences(
        user,
        now=now,
      )

    for store_id, score in store_preferences.items():
        if not np.isfinite(score) or not 0 <= score <= 1:
            raise ValueError(
                f"점포 {store_id} 선호도는 0~1이어야 합니다."
            )

    # ① 관심 상품 일치
    df["interest_match"] = (
        df["product_id"]
        .isin(profile["interest_product_ids"])
        .astype(float)
    )

    # ② 카테고리 선호도
    df["category_preference"] = (
        df["category"]
        .map(category_preferences)
        .fillna(0.0)
        .astype(float)
    )

    # ③ 점포 선호도
    df["store_preference"] = (
        df["store_id"]
        .map(store_preferences)
        .fillna(0.0)
        .astype(float)
    )

    # ④ 거리: 가까울수록 높은 점수
    df["distance_score"] = (
        1.0 / (1.0 + df["distance_km"])
    )

    # ⑤ 등록 시점: 최근일수록 높은 점수
    # 식품의 실제 신선도를 의미하지 않습니다.
    df["recency_score"] = np.exp(
        -df["hours_since_created"] / 24.0
    )

    # 최종 점수와 각 항목의 기여 점수
    df["score"] = 0.0

    for feature, weight in weights.items():
        contribution_column = f"{feature}_contribution"

        df[contribution_column] = df[feature] * weight
        df["score"] += df[contribution_column]

    # 동점이면 거리 → 최근 등록 → 게시물 ID 순
    df = df.sort_values(
        [
            "score",
            "distance_km",
            "hours_since_created",
            "listing_id",
        ],
        ascending=[False, True, True, True],
    ).reset_index(drop=True)

    if top_n is not None:
        df = df.head(top_n).copy()

    df.insert(0, "rank", range(1, len(df) + 1))

    return {
        "state": state,
        "state_name": STATE_NAMES[state],
        "weights": weights,
        "profile": profile,
        "category_preferences": category_preferences,
        "items": df,
    }


# ============================================================
# 7. 백엔드 입력 → 추천 함수용 데이터 변환
# ============================================================
def build_recommendation_inputs(payload):
    as_of = parse_aware_time(payload["as_of"])
    raw_user = payload["user"]

    # 1. 표준 상품 → 카테고리
    product_categories = {}

    def register_product_category(product_id, category_id):
        product_id = normalize_id(product_id)
        category_id = normalize_id(category_id)

        if (
            product_id in product_categories
            and product_categories[product_id] != category_id
        ):
            raise ValueError(
                f"상품 {product_id}의 카테고리가 서로 다릅니다."
            )

        product_categories[product_id] = category_id

    for product in payload["products"]:
        register_product_category(
            product["product_id"],
            product["category_id"],
        )

    # 2. 최근 30일 행동 검증·중복 제거·무효 예약 제외
    # get_recent_valid_events()가 각 행동의 store_id도
    # 반환하도록 수정되어 있어야 합니다.
    recent = get_recent_valid_events(
        raw_user.get("events", []),
        now=as_of,
    )

    # 기존 store_preferences 입력 처리 블록 삭제
    # 점포 선호도는 recommend()에서 행동 기록으로 자동 계산

    # 3. 추천 후보 상품 변환
    candidate_rows = []
    seen_listing_ids = set()

    for listing in payload["listings"]:
        listing_id = normalize_id(listing["listing_id"])

        if listing_id in seen_listing_ids:
            raise ValueError(
                f"판매 게시물이 중복되었습니다: {listing_id}"
            )

        seen_listing_ids.add(listing_id)

        product_id = normalize_id(listing["product_id"])
        category_id = normalize_id(listing["category_id"])

        register_product_category(
            product_id,
            category_id,
        )

        created_at = parse_aware_time(
            listing["created_at"]
        )

        if created_at > as_of:
            raise ValueError(
                f"상품 {listing_id}의 등록 시각이 "
                "추천 기준 시각보다 미래입니다."
            )

        candidate_rows.append({
            "listing_id": listing_id,
            "product_id": product_id,
            "name": listing["name"],
            "category": category_id,
            "store_id": normalize_id(
                listing["store_id"]
            ),
            "distance_km": listing["distance_km"],
            "hours_since_created": (
                as_of - created_at
            ).total_seconds() / 3600.0,
        })

    # 4. 추천 함수에 전달할 사용자 정보
    # store_preferences 항목 삭제
    user = {
        "user_id": normalize_id(raw_user["user_id"]),
        "interest_product_ids": [
            normalize_id(product_id)
            for product_id in raw_user.get(
                "interest_product_ids", []
            )
        ],
        "product_categories": product_categories,
        "events": recent.to_dict(orient="records"),
    }

    # 5. 관심·행동 상품의 카테고리 누락 확인
    required_product_ids = (
        set(user["interest_product_ids"])
        | set(recent["product_id"])
    )

    missing_product_ids = (
        required_product_ids - set(product_categories)
    )

    if missing_product_ids:
        raise ValueError(
            "다음 관심·행동 상품의 카테고리 정보가 없습니다: "
            f"{sorted(missing_product_ids)}"
        )

    # 6. 후보가 0개여도 동일한 컬럼 유지
    candidates = pd.DataFrame(
        candidate_rows,
        columns=CANDIDATE_COLUMNS,
    )

    return user, candidates, as_of

# ============================================================
# 8. 백엔드가 호출할 최종 함수
# ============================================================

def recommend_from_payload(payload, top_n=20):
    user, candidates, as_of = build_recommendation_inputs(
        payload
    )

    result = recommend(
        user,
        candidates,
        top_n=top_n,
        now=as_of,
    )

    items = [
        {
            "listing_id": row["listing_id"],
            "rank": int(row["rank"]),
            "score": float(row["score"]),
        }
        for row in result["items"].to_dict(
            orient="records"
        )
    ]

    return {
        "request_id": normalize_id(
            payload["request_id"]
        ),
        "generated_at": as_of.isoformat(),
        "rule_version": RULE_VERSION,
        "user_state": result["state"],
        "items": items,
    }


# ============================================================
# 9. 여기부터 테스트용 데이터
# 실제 연결 시 백엔드가 같은 형식의 payload를 생성합니다.
# ============================================================

test_now = pd.Timestamp.now(tz="Asia/Seoul")


def make_event(
    event_id,
    product_id,
    event_type,
    days_ago=1,
    **extra,
):
    return {
        "event_id": event_id,
        "product_id": product_id,
        "store_id": {"10": "S01", "11": "S01", "20": "S02", "30": "S03", "40": "S05"}[product_id],
        "event_type": event_type,
        "occurred_at": (
            test_now - pd.Timedelta(days=days_ago)
        ).isoformat(),
        **extra,
    }


base_payload = {
    "as_of": test_now.isoformat(),
    "request_id": "rec-example",

    "user": {
        "user_id": "U001",
        "interest_product_ids": [],
        "events": [],

    },

    # 판매 후보에 없는 과거 행동 상품도 포함
    "products": [
        {"product_id": "10", "category_id": "fruit"},
        {"product_id": "11", "category_id": "fruit"},
        {"product_id": "20", "category_id": "seafood"},
        {"product_id": "30", "category_id": "meat"},
        {"product_id": "40", "category_id": "vegetable"},
    ],

    # 백엔드가 판매 중·시장·거리 조건을 확인했다고 가정
    "listings": [
        {
            "listing_id": "L101",
            "product_id": "10",
            "store_id": "S01",
            "name": "사과 1kg",
            "category_id": "fruit",
            "distance_km": 0.5,
            "created_at": (
                test_now - pd.Timedelta(hours=2)
            ).isoformat(),
        },
        {
            "listing_id": "L102",
            "product_id": "20",
            "store_id": "S02",
            "name": "고등어 2마리",
            "category_id": "seafood",
            "distance_km": 0.2,
            "created_at": (
                test_now - pd.Timedelta(hours=1)
            ).isoformat(),
        },
        {
            "listing_id": "L103",
            "product_id": "30",
            "store_id": "S03",
            "name": "삼겹살 600g",
            "category_id": "meat",
            "distance_km": 1.2,
            "created_at": (
                test_now - pd.Timedelta(hours=3)
            ).isoformat(),
        },
        {
            "listing_id": "L104",
            "product_id": "10",
            "store_id": "S04",
            "name": "다른 점포의 사과 1kg",
            "category_id": "fruit",
            "distance_km": 1.5,
            "created_at": (
                test_now - pd.Timedelta(minutes=30)
            ).isoformat(),
        },
    ],
}


# A. 초기 사용자
payload_a = copy.deepcopy(base_payload)
payload_a["request_id"] = "rec-U001"


# B. 서로 다른 상품 2개 찜 → 관심 행동 사용자
payload_b = copy.deepcopy(base_payload)
payload_b["request_id"] = "rec-U002"
payload_b["user"] = {
    "user_id": "U002",
    "interest_product_ids": ["10"],

    "events": [
        make_event("B01", "10", "favorite"),
        make_event("B02", "20", "favorite"),
    ],
}


# C. 상세 조회 5개 + 유효 예약·구매 상품 5개 → 기본 사용자
payload_c = copy.deepcopy(base_payload)
payload_c["request_id"] = "rec-U003"

view_events = [
    make_event(
        f"C-view-{product_id}",
        product_id,
        "detail_view",
        days_ago=3,
    )
    for product_id in ["10", "11", "20", "30", "40"]
]

conversion_events = [
    make_event(
        "C-reserve-10",
        "10",
        "reserve",
        days_ago=2,
        is_valid_reservation=True,
    ),
    make_event(
        "C-reserve-11",
        "11",
        "reserve",
        days_ago=2,
        is_valid_reservation=True,
    ),
    make_event(
        "C-purchase-20",
        "20",
        "purchase_confirmed",
    ),
    make_event(
        "C-purchase-30",
        "30",
        "purchase_confirmed",
    ),
    make_event(
        "C-purchase-40",
        "40",
        "purchase_confirmed",
    ),
]

payload_c["user"] = {
    "user_id": "U003",
    "interest_product_ids": ["20"],
    "events": view_events + conversion_events,

    # 점포 선호도는 아직 테스트용 값을 직접 입력
    "store_preferences": [
        {"store_id": "S01", "score": 0.1},
        {"store_id": "S02", "score": 0.9},
        {"store_id": "S03", "score": 0.2},
        {"store_id": "S04", "score": 0.0},
    ],
}


# ============================================================
# 10. 테스트 실행
# ============================================================

if __name__ == "__main__":
    expected_states = [
        "new_no_interest",
        "new_with_interest",
        "behavior_based",
    ]

    for payload, expected_state in zip(
        [payload_a, payload_b, payload_c],
        expected_states,
    ):
        # 백엔드가 호출할 함수
        response = recommend_from_payload(
            payload,
            top_n=20,
        )

        assert response["user_state"] == expected_state

        # 아래는 계산 과정 확인용
        user, candidates, as_of = (
            build_recommendation_inputs(payload)
        )

        details = recommend(
            user,
            candidates,
            top_n=20,
            now=as_of,
        )

        print("\n" + "=" * 60)
        print("사용자:", user["user_id"])
        print("상태:", details["state_name"])
        print("가중치:", details["weights"])

        profile = details["profile"]

        print(
            "상세 조회 상품 수:",
            profile["viewed_product_count"],
        )
        print(
            "찜·유효 예약 상품 수:",
            profile[
                "favorite_or_reserved_product_count"
            ],
        )
        print(
            "유효 예약·확인 구매 상품 수:",
            profile[
                "reserved_or_purchased_product_count"
            ],
        )
        print(
            "카테고리 선호도:",
            details["category_preferences"],
        )

        print("\n정렬 결과:")
        print(
            details["items"][
                [
                    "rank",
                    "listing_id",
                    "name",
                    "interest_match",
                    "category_preference",
                    "store_preference",
                    "distance_score",
                    "recency_score",
                    "score",
                ]
            ].round(4).to_string(index=False)
        )

        print("\n백엔드 반환 결과:")
        print(
            json.dumps(
                response,
                ensure_ascii=False,
                indent=2,
            )
        )

    print("\n세 사용자 상태 테스트 완료!")

In [ ]:
#신규 사용자, 기존 사용자 판단이 잘 되는지 확인하는 코드


profile = get_user_recommendation_profile(user)

print("적용 상태:", STATE_NAMES[profile["state"]])
print("상세 조회 상품 수:", profile["viewed_product_count"])
print(
    "찜·유효 예약 상품 수:",
    profile["favorite_or_reserved_product_count"],
)
print(
    "유효 예약·확인 구매 상품 수:",
    profile["reserved_or_purchased_product_count"],
)

result = recommend(user, candidates, top_n=20)

print("가중치:", result["weights"])

display(
    result["items"][
        ["rank", "listing_id", "name", "score"]
    ]
)

In [ ]:
#
# **AI LightGBM Ranker를 이용한 학습**
#

!pip -q install lightgbm joblib
!pip -q install lightgbm pandas numpy scikit-learn joblib

In [ ]:
# 학습용 데이터가 준비된 뒤 사용할 코드의 형태

import numpy as np
import pandas as pd
import joblib
from lightgbm import LGBMRanker

rng = np.random.default_rng(42)
rows = []

# 가상의 추천 요청 200개, 요청마다 상품 10개
for request_num in range(200):
    requested_at = (
        pd.Timestamp("2026-01-01", tz="UTC")
        + pd.Timedelta(hours=request_num)
    )

    for item_num in range(10):
        interest_match = int(rng.integers(0, 2))
        category_preference = float(rng.random())
        store_preference = float(rng.random())
        distance_km = float(rng.uniform(0.1, 5.0))
        hours_since_created = float(rng.uniform(0, 48))

        # 실행 확인용 가상 정답을 만드는 계산입니다.
        # 실제 서비스에서는 노출 후 클릭·찜·예약 등으로 정답을 만듭니다.
        simulated_score = (
            2.0 * interest_match
            + category_preference
            + 0.5 * store_preference
            - 0.3 * distance_km
            + rng.normal(0, 0.5)
        )

        label = int(
            np.digitize(simulated_score, bins=[0.0, 1.0, 2.0, 3.0])
        )

        rows.append({
            "request_id": f"R{request_num:04d}",
            "requested_at": requested_at,
            "listing_id": item_num + 1,
            "interest_match": interest_match,
            "category_preference": category_preference,
            "store_preference": store_preference,
            "distance_km": distance_km,
            "hours_since_created": hours_since_created,
            "label": label,
        })

df = pd.DataFrame(rows)

# 과거 요청 160개는 학습, 이후 요청 40개는 검증에 사용합니다.
# 같은 요청의 상품들은 같은 구간에 들어갑니다.
cutoff = df["requested_at"].sort_values().unique()[160]

train_df = df.loc[df["requested_at"] < cutoff].copy()
valid_df = df.loc[df["requested_at"] >= cutoff].copy()

# 요청별 상품이 연속해서 배치되도록 정렬
train_df = train_df.sort_values(
    ["request_id", "listing_id"]
).reset_index(drop=True)

valid_df = valid_df.sort_values(
    ["request_id", "listing_id"]
).reset_index(drop=True)

features = [
    "interest_match",
    "category_preference",
    "store_preference",
    "distance_km",
    "hours_since_created",
]

train_groups = (
    train_df.groupby("request_id", sort=False).size().to_list()
)
valid_groups = (
    valid_df.groupby("request_id", sort=False).size().to_list()
)

print("학습 데이터:", train_df.shape)
print("검증 데이터:", valid_df.shape)
print("학습 요청 수:", len(train_groups))
print("검증 요청 수:", len(valid_groups))

model = LGBMRanker(
    objective="lambdarank",
    n_estimators=100,
    learning_rate=0.05,
    num_leaves=15,
    random_state=42,
    verbosity=-1,
)


# 학습이 되는 부분 코드 (코드로 만든 가상의 반응 패턴을 배운 연습용 모델)
model.fit(
    train_df[features],
    train_df["label"],
    group=train_groups,
    eval_set=[(
        valid_df[features],
        valid_df["label"],
    )],
    eval_group=[valid_groups],
    eval_at=[5, 10],
)

joblib.dump(
    {
        "model": model,
        "features": features,
        "version": "demo_ranker_v1",
    },
    "recommendation_model.joblib",
)

print("\n학습 및 모델 저장 완료!")
print("검증 지표:", model.best_score_)

In [ ]:
# 위 셀의 이중 for 문으로 만들어진 가상 데이터 눈으로 확인하기
#

print("전체 행·열 수:", df.shape)
display(df.head(20))  # 처음 20행 보기

In [ ]:
#
# 검증 데이터에서 요청 하나를 골라 그 요청의 상품들을 정렬 = 사용자에게 노출되는 순서!
#

example_request_id = valid_df["request_id"].iloc[0]

example = valid_df.loc[
    valid_df["request_id"] == example_request_id
].copy()

#학습한 모델을 사용하는 예측 과정
example["score"] = model.predict(example[features])

#모델이 상품별 점수를 계산하고, 이 코드 대로 점수가 정렬됨
ranked = example.sort_values("score", ascending=False)

display(
    ranked[
        [
            "listing_id",
            "interest_match",
            "category_preference",
            "distance_km",
            "score",
            "label",
        ]
    ]
)

In [ ]:
#
# 기존 검증 결과를 확인
# 검증 데이터 400행으로 NDCG@5, NDCG@10을 계산
#


print(model.best_score_)
# NDCG는 정답 등급(label)이 높은 상품을 목록 앞에 배치했는지 평가하는 지표입니다.
# - NDCG@5: 상위 5개 상품의 순서 평가
# - NDCG@10: 상위 10개 상품의 순서 평가
# - 값은 0~1이며 높을수록 정답 순서에 가깝습니다.

In [ ]:
#
#같은 검증 요청별로 세 가지 정렬 방법을 비교
#

import numpy as np
import pandas as pd
from sklearn.metrics import ndcg_score

evaluated = valid_df.copy()

evaluated["model_score"] = model.predict(evaluated[features])

# 가까운 상품일수록 높은 점수
evaluated["distance_score"] = -evaluated["distance_km"]

rng = np.random.default_rng(123)
evaluated["random_score"] = rng.random(len(evaluated))

def mean_ndcg(data, score_column, k):
    values = []

    for _, group in data.groupby("request_id"):
        labels = group["label"].to_numpy()

        # 반응 등급이 전부 0인 요청은 순위 평가에서 제외
        if labels.max() == 0:
            continue

        # 앞선 LightGBM 기본 설정과 같은 등급별 이득 사용
        gains = (2.0 ** labels) - 1.0

        values.append(
            ndcg_score(
                gains.reshape(1, -1),
                group[score_column].to_numpy().reshape(1, -1),
                k=k,
            )
        )

    return float(np.mean(values)) if values else float("nan")

results = []

for name, column in [
    ("무작위 정렬", "random_score"),
    ("거리순 정렬", "distance_score"),
    ("학습 모델", "model_score"),
]:
    results.append({
        "방법": name,
        "NDCG@5": mean_ndcg(evaluated, column, 5),
        "NDCG@10": mean_ndcg(evaluated, column, 10),
    })

display(pd.DataFrame(results).round(4))

In [ ]:
# 과적합 판단 코드

model.fit(
    train_df[features],
    train_df["label"],
    group=train_groups,
    eval_set=[
        (train_df[features], train_df["label"]),
        (valid_df[features], valid_df["label"]),
    ],
    eval_names=["train", "valid"],
    eval_group=[train_groups, valid_groups],
    eval_at=[5, 10],
)

print("학습 데이터:", model.best_score_["train"])
print("검증 데이터:", model.best_score_["valid"])

In [ ]:
# ============================================================
# 장보소: 실제 데이터용 LightGBM 추천 모델 학습
#
# 전제:
# - 실제 노출된 상품만 학습용 CSV에 포함
# - 입력값은 추천 당시 값
# - label은 이후 행동을 별도 집계해서 확정한 값
# - 같은 행동의 중복 귀속은 CSV 생성 단계에서 처리
# ============================================================

from pathlib import Path
import json

import joblib
import lightgbm as lgb
import numpy as np
import pandas as pd

from sklearn.metrics import ndcg_score


# ============================================================
# 1. 설정 — 실제 데이터에 맞게 수정
# ============================================================

DATA_PATH = "/content/recommendation_training_data.csv"
OUTPUT_DIR = Path("/content/jangboso_model")

# 아래 날짜는 예시입니다. 실제 수집 기간에 맞게 바꾸세요.
VALID_START = "2026-11-06T00:00:00+09:00"
TEST_START = "2026-11-08T00:00:00+09:00"

# 이 시점까지 수집·확인된 기록만 사용합니다.
# 현재 실제 보유 데이터의 추출 기준 시각으로 지정하세요.
DATA_AS_OF = "2026-11-09T00:00:00+09:00"

MODEL_VERSION = "ranker_v1"
FEATURE_DEFINITION_VERSION = "features_v1"
LABEL_DEFINITION_VERSION = "labels_v1"

# label 0~4에 부여할 가치
# 초기 설정이며, 조정은 검증 데이터로만 수행합니다.
LABEL_GAIN = [0, 1, 3, 5, 8]

FEATURES = [
    "interest_match",
    "category_preference",
    "store_preference",
    "distance_km",
    "hours_since_created",
    "is_interest_user",
    "is_behavior_user",
]

STATE_NAMES = [
    "new_no_interest",
    "new_with_interest",
    "behavior_based",
]

REQUIRED_COLUMNS = [
    "request_id",
    "user_id",
    "listing_id",
    "requested_at",
    "impressed_at",
    "label_available_at",
    "interest_match",
    "category_preference",
    "store_preference",
    "distance_km",
    "hours_since_created",
    "user_state",
    "rule_score",
    "label",
]


# ============================================================
# 2. 시간 처리
# ============================================================

def parse_aware_time(value):
    if value is None or pd.isna(value):
        raise ValueError("시각이 비어 있습니다.")

    timestamp = pd.Timestamp(value)

    if timestamp.tzinfo is None:
        raise ValueError(
            "시각에는 시간대가 필요합니다. "
            "예: 2026-12-01T00:00:00+09:00"
        )

    return timestamp.tz_convert("UTC")


# ============================================================
# 3. 데이터 읽기·검사
# ============================================================

def load_training_data(path):
    df = pd.read_csv(
        path,
        dtype={
            "request_id": "string",
            "user_id": "string",
            "listing_id": "string",
            "user_state": "string",
        },
    )

    missing = sorted(set(REQUIRED_COLUMNS) - set(df.columns))

    if missing:
        raise ValueError(f"필수 컬럼이 없습니다: {missing}")

    if df.empty:
        raise ValueError("학습 데이터가 비어 있습니다.")

    if df[REQUIRED_COLUMNS].isna().any().any():
        bad_columns = (
            df[REQUIRED_COLUMNS]
            .columns[df[REQUIRED_COLUMNS].isna().any()]
            .tolist()
        )
        raise ValueError(f"결측값이 있는 컬럼: {bad_columns}")

    for column in [
        "request_id",
        "user_id",
        "listing_id",
        "user_state",
    ]:
        df[column] = df[column].str.strip()

        if df[column].eq("").any():
            raise ValueError(f"{column}에 빈 문자열이 있습니다.")

    for column in [
        "requested_at",
        "impressed_at",
        "label_available_at",
    ]:
        df[column] = pd.to_datetime(
            [parse_aware_time(value) for value in df[column]],
            utc=True,
        )

    numeric_columns = [
        "interest_match",
        "category_preference",
        "store_preference",
        "distance_km",
        "hours_since_created",
        "rule_score",
        "label",
    ]

    for column in numeric_columns:
        df[column] = pd.to_numeric(
            df[column],
            errors="raise",
        )

        if not np.isfinite(
            df[column].to_numpy(dtype=float)
        ).all():
            raise ValueError(f"{column}에 유효하지 않은 숫자가 있습니다.")

    if not df["interest_match"].isin([0, 1]).all():
        raise ValueError("interest_match는 0 또는 1이어야 합니다.")

    for column in [
        "category_preference",
        "store_preference",
        "rule_score",
    ]:
        if not df[column].between(0, 1).all():
            raise ValueError(f"{column}은 0~1이어야 합니다.")

    for column in ["distance_km", "hours_since_created"]:
        if (df[column] < 0).any():
            raise ValueError(f"{column}은 0 이상이어야 합니다.")

    if not df["label"].isin(range(len(LABEL_GAIN))).all():
        raise ValueError("label은 0~4 정수여야 합니다.")

    df["label"] = df["label"].astype(int)

    if not df["user_state"].isin(STATE_NAMES).all():
        raise ValueError("알 수 없는 user_state가 있습니다.")

    if df.duplicated(["request_id", "listing_id"]).any():
        raise ValueError(
            "동일 request_id·listing_id가 중복되었습니다. "
            "CSV 생성 단계에서 반복 노출을 정리해 주세요."
        )

    if (df["impressed_at"] < df["requested_at"]).any():
        raise ValueError("추천 요청보다 앞선 노출 시각이 있습니다.")

    if (df["label_available_at"] <= df["impressed_at"]).any():
        raise ValueError("정답 확정 시각은 노출 시각보다 뒤여야 합니다.")

    # 한 요청의 사용자·요청 시각·사용자 상태는 같아야 함
    for column in ["user_id", "requested_at", "user_state"]:
        counts = df.groupby("request_id")[column].nunique()

        if counts.gt(1).any():
            raise ValueError(
                f"한 request_id 안에서 {column} 값이 서로 다릅니다."
            )

    # 서버에서도 동일하게 생성해야 하는 상태 입력값
    df["is_interest_user"] = (
        df["user_state"] == "new_with_interest"
    ).astype(int)

    df["is_behavior_user"] = (
        df["user_state"] == "behavior_based"
    ).astype(int)

    return df


# ============================================================
# 4. 시간 기준 학습·검증·테스트 분리
# ============================================================

def split_by_time(df):
    valid_start = parse_aware_time(VALID_START)
    test_start = parse_aware_time(TEST_START)
    data_as_of = parse_aware_time(DATA_AS_OF)

    if not valid_start < test_start < data_as_of:
        raise ValueError(
            "VALID_START < TEST_START < DATA_AS_OF여야 합니다."
        )

    # 요청별로 가장 늦은 정답 확정 시각 사용
    # 요청 중 일부 상품만 다른 구간으로 갈라지는 것을 방지
    requests = df.groupby("request_id").agg(
        requested_at=("requested_at", "first"),
        label_available_at=("label_available_at", "max"),
    )

    train_ids = requests.index[
        (requests["requested_at"] < valid_start)
        & (requests["label_available_at"] < valid_start)
    ]

    valid_ids = requests.index[
        (requests["requested_at"] >= valid_start)
        & (requests["requested_at"] < test_start)
        & (requests["label_available_at"] < test_start)
    ]

    test_ids = requests.index[
        (requests["requested_at"] >= test_start)
        & (requests["requested_at"] <= data_as_of)
        & (requests["label_available_at"] <= data_as_of)
    ]

    parts = []

    for name, ids in [
        ("train", train_ids),
        ("valid", valid_ids),
        ("test", test_ids),
    ]:
        part = df.loc[df["request_id"].isin(ids)].copy()

        part = part.sort_values(
            ["request_id", "listing_id"]
        ).reset_index(drop=True)

        if part.empty:
            raise ValueError(
                f"{name} 데이터가 비어 있습니다. "
                "날짜 설정과 데이터 수집 기간을 확인하세요."
            )

        print(
            f"{name}: {len(part):,}행 / "
            f"{part['request_id'].nunique():,}요청 / "
            f"{part['user_id'].nunique():,}사용자"
        )

        parts.append(part)

    used_requests = sum(
        part["request_id"].nunique() for part in parts
    )

    print(
        "경계·미확정 등으로 제외한 요청:",
        len(requests) - used_requests,
    )

    return tuple(parts)


# ============================================================
# 5. 학습 가능한 요청 선정·그룹 생성
# ============================================================

def select_informative_training_requests(df):
    stats = df.groupby("request_id").agg(
        size=("label", "size"),
        distinct_labels=("label", "nunique"),
    )

    usable_ids = stats.index[
        (stats["size"] >= 2)
        & (stats["distinct_labels"] >= 2)
    ]

    result = df.loc[
        df["request_id"].isin(usable_ids)
    ].copy()

    result = result.sort_values(
        ["request_id", "listing_id"]
    ).reset_index(drop=True)

    print(
        "순위 학습에 사용하는 요청:",
        f"{len(usable_ids):,} / {len(stats):,}",
    )

    if result.empty:
        raise ValueError(
            "서로 다른 정답 등급을 가진 학습 요청이 없습니다."
        )

    return result


def group_sizes(df):
    groups = (
        df.groupby("request_id", sort=False)
        .size()
        .to_numpy()
    )

    if int(groups.sum()) != len(df):
        raise ValueError("그룹 크기 합계가 행 수와 다릅니다.")

    return groups


# ============================================================
# 6. NDCG 평가
# - 모델 label_gain과 같은 가치 배열 사용
# - 상품이 2개 이상이고 양의 정답이 있는 요청을 평가
# - 모든 등급이 같은 경우 순위 판별 정보가 적다는 점에 유의
# ============================================================

def evaluate_rankings(df, score_columns):
    gain_array = np.asarray(LABEL_GAIN, dtype=float)

    eligible_groups = []

    for _, group in df.groupby("request_id", sort=False):
        if len(group) < 2 or group["label"].max() == 0:
            continue

        eligible_groups.append(group)

    if not eligible_groups:
        raise ValueError(
            "NDCG를 평가할 요청이 없습니다. "
            "노출 수와 반응 기록을 확인하세요."
        )

    rows = []

    for method, score_column in score_columns.items():
        scores_at_5 = []
        scores_at_10 = []

        for group in eligible_groups:
            gains = gain_array[
                group["label"].to_numpy(dtype=int)
            ].reshape(1, -1)

            predictions = group[
                score_column
            ].to_numpy(dtype=float).reshape(1, -1)

            scores_at_5.append(
                ndcg_score(gains, predictions, k=5)
            )
            scores_at_10.append(
                ndcg_score(gains, predictions, k=10)
            )

        rows.append({
            "method": method,
            "NDCG@5": float(np.mean(scores_at_5)),
            "NDCG@10": float(np.mean(scores_at_10)),
            "evaluated_requests": len(eligible_groups),
            "total_requests": df["request_id"].nunique(),
        })

    return pd.DataFrame(rows)


def add_model_predictions(df, model):
    result = df.copy()

    result["model_score"] = model.predict(
        result[FEATURES],
        num_iteration=model.best_iteration_,
    )

    result["distance_score"] = -result["distance_km"]

    return result


# ============================================================
# 7. 데이터 준비
# ============================================================

df = load_training_data(DATA_PATH)

train_all, valid_df, test_df = split_by_time(df)

# 학습에서만 순위 정보가 없는 요청 제외
train_df = select_informative_training_requests(train_all)

# 검증 구간에 평가 가능한 요청이 있는지 먼저 검사
evaluate_rankings(
    valid_df,
    {"규칙 추천": "rule_score"},
)

train_groups = group_sizes(train_df)
valid_groups = group_sizes(valid_df)


# ============================================================
# 8. 모델 학습
# ============================================================

model = lgb.LGBMRanker(
    objective="lambdarank",
    metric="ndcg",
    label_gain=LABEL_GAIN,

    n_estimators=1000,
    learning_rate=0.03,
    num_leaves=15,
    min_child_samples=20,
    reg_lambda=1.0,

    random_state=42,
    verbosity=-1,
)

model.fit(
    train_df[FEATURES],
    train_df["label"],
    group=train_groups,

    eval_set=[
        (valid_df[FEATURES], valid_df["label"]),
    ],
    eval_group=[valid_groups],
    eval_names=["valid"],
    eval_at=[5, 10],

    callbacks=[
        lgb.early_stopping(
            stopping_rounds=50,
            first_metric_only=True,
        ),
        lgb.log_evaluation(period=50),
    ],
)

print("\n선택된 트리 수:", model.best_iteration_)
print("LightGBM 내부 검증 지표:", model.best_score_)

# 내부 지표와 아래 직접 계산하는 지표는
# 평가 요청 선정·동점 처리 차이로 값이 다를 수 있습니다.
# 방법 간 비교는 아래 공통 평가 함수의 결과로 판단합니다.


# ============================================================
# 9. 검증 데이터에서 규칙 추천과 비교
# ============================================================

METHODS = {
    "거리순": "distance_score",
    "규칙 추천": "rule_score",
    "학습 모델": "model_score",
}

valid_scored = add_model_predictions(valid_df, model)

valid_report = evaluate_rankings(
    valid_scored,
    METHODS,
)

print("\n검증 구간 비교")
print(valid_report.round(4).to_string(index=False))


# ============================================================
# 10. 최종 테스트
#
# 이 결과를 보고 설정을 반복 변경하면 테스트 구간도
# 사실상 검증 데이터가 됩니다.
# 설정 확정 후 최종 확인 용도로 사용하세요.
# ============================================================

test_scored = add_model_predictions(test_df, model)

test_report = evaluate_rankings(
    test_scored,
    METHODS,
)

print("\n최종 테스트 구간 비교")
print(test_report.round(4).to_string(index=False))

print("\n사용자 상태별 최종 테스트")

for state in STATE_NAMES:
    subset = test_scored.loc[
        test_scored["user_state"] == state
    ]

    if subset.empty:
        print(f"\n{state}: 데이터 없음")
        continue

    try:
        report = evaluate_rankings(subset, METHODS)
        print(f"\n{state}")
        print(report.round(4).to_string(index=False))
    except ValueError as error:
        print(f"\n{state}: {error}")


# ============================================================
# 11. 학습 데이터에서 점수 확인 — 과적합 참고용
# ============================================================

train_scored = add_model_predictions(train_df, model)

train_report = evaluate_rankings(
    train_scored,
    {"학습 모델": "model_score"},
)

print("\n학습 구간 참고 지표")
print(train_report.round(4).to_string(index=False))


# ============================================================
# 12. 모델·메타데이터·평가 결과 저장
#
# 저장했다고 자동으로 서비스에 적용되는 것은 아닙니다.
# ============================================================

OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

metadata = {
    "version": MODEL_VERSION,
    "feature_definition_version": FEATURE_DEFINITION_VERSION,
    "label_definition_version": LABEL_DEFINITION_VERSION,

    "features": FEATURES,
    "label_gain": LABEL_GAIN,
    "best_iteration": int(model.best_iteration_),

    "lightgbm_version": lgb.__version__,
    "pandas_version": pd.__version__,
    "numpy_version": np.__version__,

    "trained_at": pd.Timestamp.now(tz="UTC").isoformat(),

    "valid_start": VALID_START,
    "test_start": TEST_START,
    "data_as_of": DATA_AS_OF,

    "label_meanings": {
        "0": "관찰 기간 내 반응 없음",
        "1": "상세 조회",
        "2": "찜",
        "3": "유효 예약",
        "4": "확인된 구매",
    },
}

joblib.dump(
    {
        "model": model,
        **metadata,
    },
    OUTPUT_DIR / "recommendation_model.joblib",
)

with open(
    OUTPUT_DIR / "metadata.json",
    "w",
    encoding="utf-8",
) as file:
    json.dump(
        metadata,
        file,
        ensure_ascii=False,
        indent=2,
    )

valid_report.to_csv(
    OUTPUT_DIR / "validation_metrics.csv",
    index=False,
    encoding="utf-8-sig",
)

test_report.to_csv(
    OUTPUT_DIR / "test_metrics.csv",
    index=False,
    encoding="utf-8-sig",
)

print("\n저장 위치:", OUTPUT_DIR)


# ============================================================
# 13. 저장된 모델을 불러와 상품 정렬
#
# 실제 서버에서는 학습 때와 같은 계산 코드로
# FEATURES 컬럼들을 준비해서 전달해야 합니다.
# ============================================================

def rank_with_saved_model(candidate_features, model_path):
    bundle = joblib.load(model_path)

    feature_names = bundle["features"]
    loaded_model = bundle["model"]

    required = ["listing_id"] + feature_names
    missing = sorted(
        set(required) - set(candidate_features.columns)
    )

    if missing:
        raise ValueError(f"예측 입력에 없는 컬럼: {missing}")

    result = candidate_features.copy()

    if result["listing_id"].duplicated().any():
        raise ValueError("예측 후보 listing_id가 중복되었습니다.")

    if result.empty:
        result["score"] = pd.Series(dtype=float)
        result["rank"] = pd.Series(dtype=int)
        return result

    feature_values = (
        result[feature_names]
        .apply(pd.to_numeric, errors="raise")
        .to_numpy(dtype=float)
    )

    if not np.isfinite(feature_values).all():
        raise ValueError("예측 입력에 결측값 또는 무한대가 있습니다.")

    result["score"] = loaded_model.predict(
        result[feature_names],
        num_iteration=bundle["best_iteration"],
    )

    result = result.sort_values(
        ["score", "listing_id"],
        ascending=[False, True],
    ).reset_index(drop=True)

    result["rank"] = np.arange(1, len(result) + 1)

    return result


# 테스트 구간에서 요청 하나를 골라 예측 흐름 확인
example_request_id = test_df["request_id"].iloc[0]

example_candidates = test_df.loc[
    test_df["request_id"] == example_request_id,
    ["listing_id"] + FEATURES,
].copy()

ranked = rank_with_saved_model(
    example_candidates,
    OUTPUT_DIR / "recommendation_model.joblib",
)

print("\n요청 하나의 모델 추천 순서")
print(
    ranked[
        ["rank", "listing_id", "score"]
    ].round(4).to_string(index=False)
)